# Notebook 06: Eskom Gold Materialized Lake Views

This notebook creates Fabric Materialized Lake Views in the schema-enabled Gold Lakehouse `lh_gld_finance`.

**Attach `lh_gld_finance` as the default Lakehouse for this notebook.**

The notebook uses the correct Spark SQL syntax:

```sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW ... AS SELECT ...
```

The views source only from the fine-grained Gold facts and do not read Bronze or Silver.

Created objects:

- `mlv.finance_actual_monthly`
- `mlv.vendor_spend_monthly`
- `mlv.gl_balance_monthly`
- `mlv.plan_vs_actual_monthly`

The plan-versus-actual view uses a `UNION ALL` followed by aggregation. This avoids duplicating actuals across plan versions and keeps `PLAN_VERSION` and `PLAN_SOURCE` explicit.

In [ ]:
# Validate that the Gold Lakehouse is the notebook default.
from pyspark.sql import functions as F

expected_lakehouse = "lh_gld_finance"
print(f"Expected default Lakehouse: {expected_lakehouse}")
print(f"Current Spark database: {spark.catalog.currentDatabase()}")

required_tables = [
    "fct.finance_transaction",
    "fct.budget",
]

missing = []
for table_name in required_tables:
    try:
        count = spark.table(table_name).limit(1).count()
        print(f"Validated source: {table_name}")
    except Exception as error:
        missing.append(f"{table_name}: {error}")

if missing:
    raise RuntimeError(
        "Required Gold source tables are unavailable:\n" + "\n".join(missing)
    )


In [ ]:
%%sql
CREATE SCHEMA IF NOT EXISTS mlv;

-- Change Data Feed helps Fabric choose an optimal refresh strategy where supported.
ALTER TABLE fct.finance_transaction
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

ALTER TABLE fct.budget
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

## 1. Monthly Finance Actuals

Grain:

```text
Company + Division + Profit Centre + Cost Centre + GL Account + Fiscal Year + Fiscal Period
```

This is the reusable monthly actuals serving object for Finance reporting.

In [ ]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW mlv.finance_actual_monthly
(
    CONSTRAINT valid_fiscal_period
    CHECK (fiscal_period BETWEEN 1 AND 16)
    ON MISMATCH FAIL
)
PARTITIONED BY (fiscal_year)
COMMENT "Eskom monthly SAP Finance actuals aggregated from the Gold ACDOCA line-item fact"
AS
SELECT
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    CAST(gjahr AS INT) AS fiscal_year,
    CAST(poper AS INT) AS fiscal_period,
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT) AS fiscal_period_key,
    SUM(amount_local) AS actual_amount_local,
    SUM(amount_group) AS actual_amount_group,
    SUM(amount_transaction) AS actual_amount_transaction,
    SUM(quantity) AS total_quantity,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT belnr) AS document_count
FROM fct.finance_transaction
GROUP BY
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    CAST(gjahr AS INT),
    CAST(poper AS INT),
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT);

## 2. Monthly Vendor Spend

Grain:

```text
Company + Division + Vendor + GL Account + Fiscal Year + Fiscal Period
```

Only Finance lines with a resolved `vendor_sk` are included.

In [ ]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW mlv.vendor_spend_monthly
(
    CONSTRAINT vendor_required
    CHECK (vendor_sk IS NOT NULL)
    ON MISMATCH DROP,
    CONSTRAINT valid_fiscal_period
    CHECK (fiscal_period BETWEEN 1 AND 16)
    ON MISMATCH FAIL
)
PARTITIONED BY (fiscal_year)
COMMENT "Eskom monthly vendor spend derived from the Gold SAP Finance transaction fact"
AS
SELECT
    company_sk,
    division_sk,
    vendor_sk,
    gl_account_sk,
    CAST(gjahr AS INT) AS fiscal_year,
    CAST(poper AS INT) AS fiscal_period,
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT) AS fiscal_period_key,
    SUM(amount_local) AS spend_amount_local,
    SUM(amount_group) AS spend_amount_group,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT belnr) AS document_count,
    AVG(ABS(amount_local)) AS average_line_item_value
FROM fct.finance_transaction
WHERE vendor_sk IS NOT NULL
GROUP BY
    company_sk,
    division_sk,
    vendor_sk,
    gl_account_sk,
    CAST(gjahr AS INT),
    CAST(poper AS INT),
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT);

## 3. Monthly GL Movement

This view deliberately exposes period debit, credit and net movement. Opening and closing balances should be calculated in the semantic model or in a separate view once the agreed Eskom account-sign and carry-forward rules have been confirmed.

In [ ]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW mlv.gl_balance_monthly
(
    CONSTRAINT valid_fiscal_period
    CHECK (fiscal_period BETWEEN 1 AND 16)
    ON MISMATCH FAIL
)
PARTITIONED BY (fiscal_year)
COMMENT "Eskom monthly GL debit, credit and net movement derived from the Gold Finance transaction fact"
AS
SELECT
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    CAST(gjahr AS INT) AS fiscal_year,
    CAST(poper AS INT) AS fiscal_period,
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT) AS fiscal_period_key,
    SUM(CASE WHEN drcrk = 'S' THEN ABS(amount_local) ELSE CAST(0 AS DECIMAL(23,2)) END) AS debit_amount_local,
    SUM(CASE WHEN drcrk = 'H' THEN ABS(amount_local) ELSE CAST(0 AS DECIMAL(23,2)) END) AS credit_amount_local,
    SUM(amount_local) AS net_movement_local,
    COUNT(*) AS line_item_count,
    COUNT(DISTINCT belnr) AS document_count
FROM fct.finance_transaction
GROUP BY
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    CAST(gjahr AS INT),
    CAST(poper AS INT),
    (CAST(gjahr AS INT) * 100) + CAST(poper AS INT);

## 4. Monthly Plan versus Actual

The view standardizes Actual, Budget, Forecast and FAGLFLEXP Plan values into one monthly reporting grain.

For actual rows, `plan_source = 'ACTUAL'`, `scenario = 'ACTUAL'`, and `plan_version = 'ACTUAL'`.

For planning rows, the values from `fct.budget` are retained, including FAGLFLEXP versions such as F1 through F5.

In [ ]:
%%sql
CREATE OR REPLACE MATERIALIZED LAKE VIEW mlv.plan_vs_actual_monthly
(
    CONSTRAINT valid_fiscal_period
    CHECK (fiscal_period BETWEEN 1 AND 16)
    ON MISMATCH FAIL
)
PARTITIONED BY (fiscal_year)
COMMENT "Eskom monthly plan, budget, forecast and actual serving view by conformed Finance dimensions"
AS
WITH unified_finance AS
(
    SELECT
        company_sk,
        division_sk,
        profit_center_sk,
        cost_center_sk,
        gl_account_sk,
        CAST(gjahr AS INT) AS fiscal_year,
        CAST(poper AS INT) AS fiscal_period,
        (CAST(gjahr AS INT) * 100) + CAST(poper AS INT) AS fiscal_period_key,
        'ACTUAL' AS plan_source,
        'ACTUAL' AS scenario,
        'ACTUAL' AS plan_version,
        SUM(amount_local) AS actual_amount_local,
        CAST(0 AS DECIMAL(38,2)) AS plan_amount_local,
        CAST(0 AS DECIMAL(38,2)) AS forecast_amount_local
    FROM fct.finance_transaction
    GROUP BY
        company_sk,
        division_sk,
        profit_center_sk,
        cost_center_sk,
        gl_account_sk,
        CAST(gjahr AS INT),
        CAST(poper AS INT),
        (CAST(gjahr AS INT) * 100) + CAST(poper AS INT)

    UNION ALL

    SELECT
        company_sk,
        division_sk,
        profit_center_sk,
        cost_center_sk,
        gl_account_sk,
        CAST(fiscal_year AS INT) AS fiscal_year,
        CAST(fiscal_period AS INT) AS fiscal_period,
        CAST(fiscal_period_key AS INT) AS fiscal_period_key,
        plan_source,
        scenario,
        plan_version,
        CAST(0 AS DECIMAL(38,2)) AS actual_amount_local,
        SUM(COALESCE(plan_amount_local, CAST(0 AS DECIMAL(23,2)))) AS plan_amount_local,
        SUM(COALESCE(forecast_amount_local, CAST(0 AS DECIMAL(23,2)))) AS forecast_amount_local
    FROM fct.budget
    GROUP BY
        company_sk,
        division_sk,
        profit_center_sk,
        cost_center_sk,
        gl_account_sk,
        CAST(fiscal_year AS INT),
        CAST(fiscal_period AS INT),
        CAST(fiscal_period_key AS INT),
        plan_source,
        scenario,
        plan_version
)
SELECT
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    fiscal_year,
    fiscal_period,
    fiscal_period_key,
    plan_source,
    scenario,
    plan_version,
    SUM(actual_amount_local) AS actual_amount_local,
    SUM(plan_amount_local) AS plan_amount_local,
    SUM(forecast_amount_local) AS forecast_amount_local,
    SUM(actual_amount_local) - SUM(plan_amount_local) AS variance_to_plan_amount,
    SUM(actual_amount_local) - SUM(forecast_amount_local) AS variance_to_forecast_amount
FROM unified_finance
GROUP BY
    company_sk,
    division_sk,
    profit_center_sk,
    cost_center_sk,
    gl_account_sk,
    fiscal_year,
    fiscal_period,
    fiscal_period_key,
    plan_source,
    scenario,
    plan_version;

## Validation

These checks confirm the materialized lake views can be queried after creation. Fabric persists MLV results as Delta-backed objects and manages their refresh lifecycle.

In [ ]:
# Query the created Materialized Lake Views.
mlv_tables = [
    "mlv.finance_actual_monthly",
    "mlv.vendor_spend_monthly",
    "mlv.gl_balance_monthly",
    "mlv.plan_vs_actual_monthly",
]

validation_rows = []
for table_name in mlv_tables:
    try:
        row_count = spark.table(table_name).count()
        validation_rows.append((table_name, row_count, "AVAILABLE", ""))
    except Exception as error:
        validation_rows.append((table_name, 0, "ERROR", str(error)[:500]))

validation = spark.createDataFrame(
    validation_rows,
    "materialized_lake_view string, row_count long, status string, error string"
)
display(validation)

## Refresh orchestration

The FMLV definitions are declarative. Manage recurring refresh through **Lakehouse → Materialized lake views → Manage → Schedules** with Optimal refresh enabled. Fabric derives dependency order from lineage; do not schedule recurring FMLV refresh through notebook activities.